# Notebook 04 — Síntesis e informe

**Objetivo:** integrar todos los hallazgos de los notebooks anteriores en un informe
cohesionado que responda las preguntas de investigación definidas en el notebook 00.

Este notebook hace algo diferente a los anteriores: **no genera nuevos análisis**.
Su trabajo es leer los resultados que ya existen y presentarlos de forma que alguien
sin conocimientos técnicos pueda entenderlos.

**Estructura del notebook:**
1. Recapitulación de preguntas de investigación
2. Respuesta a cada pregunta con evidencia, limitaciones y nivel de confianza
3. Convergencia de señales: qué dicen las distintas técnicas sobre los mismos actores
4. Tabla resumen de actores clave
5. Conclusiones: qué respondimos, qué queda abierto
6. Exportación: cómo generar un informe PDF o HTML

**Prerequisito:** ejecutar los notebooks 01, 02 y 03 en ese orden para generar todos los parquets.

## 1. Imports y carga de resultados

In [ ]:
import sys
from pathlib import Path
import pandas as pd
import numpy as np
import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots

PROJECT_ROOT = Path().resolve().parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.utils import RESULTS_DIR

IM_RESULTS = RESULTS_DIR / 'ironmarch'

# Cargar todos los resultados generados por notebooks anteriores
def load_if_exists(filename):
    """Intenta cargar un parquet. Si no existe, retorna un DataFrame vacío."""
    path = IM_RESULTS / filename
    if path.exists():
        df = pd.read_parquet(path)
        print(f'OK  {filename}: {len(df):,} filas')
        return df
    else:
        print(f'--  {filename}: no encontrado (ejecutar notebook correspondiente)')
        return pd.DataFrame()

print('Cargando resultados...')
posts_clean          = load_if_exists('posts_clean.parquet')
users_clean          = load_if_exists('users_clean.parquet')
structural_signals   = load_if_exists('structural_signals.parquet')
semantic_signals     = load_if_exists('semantic_signals.parquet')
ner_all              = load_if_exists('ner_results.parquet')
actor_profiles       = load_if_exists('actor_profiles.parquet')

uid_to_name = dict(zip(users_clean['userid'], users_clean['username_raw'])) if len(users_clean) else {}

## 2. Recapitulación de preguntas de investigación

En el notebook 00 definimos cuatro preguntas. Este notebook las responde.

---
**P1 — ¿Quiénes son los actores clave?**  
**P2 — ¿Hay cuentas dobles (sockpuppets)?**  
**P3 — ¿Qué eventos externos correlacionan con la actividad?**  
**P4 — ¿Hay estructura de comunidades dentro del foro?**  
---

Para cada pregunta presentamos:
- **Hallazgo:** qué encontramos
- **Evidencia:** qué datos lo soportan
- **Limitación:** qué no podemos afirmar
- **Confianza:** 🔴 baja / 🟡 media / 🟢 alta

## 3. Respuesta a P1: Actores clave

In [ ]:
# P1: Actores clave — combinamos señales de todos los notebooks
# Señal 1: betweenness centrality (notebook 02)
# Señal 2: volumen de posts (notebooks 00 y 01)
# Señal 3: perfil LLM (notebook 03)

# Tabla base: todos los usuarios con al menos una señal
if len(posts_clean) > 0:
    post_counts = posts_clean.groupby('userid').size().reset_index(name='post_count')
    actors_table = post_counts.copy()
    actors_table['username'] = actors_table['userid'].map(uid_to_name)
else:
    actors_table = pd.DataFrame()

# Agregar betweenness si existe
if len(structural_signals) > 0 and 'betweenness_public' in structural_signals.columns:
    actors_table = actors_table.merge(
        structural_signals[['userid', 'betweenness_public', 'betweenness_pm', 'degree_centrality']],
        on='userid', how='left'
    )

# Agregar señales semánticas
if len(semantic_signals) > 0 and 'hdbscan_cluster' in semantic_signals.columns:
    actors_table = actors_table.merge(
        semantic_signals[['userid', 'hdbscan_cluster', 'min_delta', 'top_topic']],
        on='userid', how='left'
    )

# Agregar perfil LLM
if len(actor_profiles) > 0 and 'role' in actor_profiles.columns:
    actors_table = actors_table.merge(
        actor_profiles[['userid', 'role', 'radicalization_level', 'confidence']],
        on='userid', how='left'
    )

# Top 20 por post_count
if len(actors_table) > 0:
    top20 = actors_table.nlargest(20, 'post_count')
    print('Top 20 actores por volumen de posts:')
    display_cols = [c for c in ['username', 'post_count', 'betweenness_public', 'role', 'radicalization_level']
                    if c in top20.columns]
    print(top20[display_cols].to_string(index=False))
else:
    print('No hay datos suficientes. Ejecutar notebooks 01, 02 y 03 primero.')

In [ ]:
# Visualización: scatter de volumen vs betweenness (los dos ejes de influencia)
if len(actors_table) > 0 and 'betweenness_public' in actors_table.columns:
    plot_df = actors_table.dropna(subset=['betweenness_public']).nlargest(50, 'post_count')

    fig = go.Figure(go.Scatter(
        x=plot_df['post_count'],
        y=plot_df['betweenness_public'],
        mode='markers+text',
        marker=dict(
            size=10,
            color=plot_df['betweenness_public'],
            colorscale='YlOrRd',
            showscale=True,
            colorbar=dict(title='Betweenness'),
        ),
        text=plot_df['username'].fillna('').astype(str),
        textposition='top center',
        textfont=dict(size=8),
        hovertext=plot_df.apply(
            lambda r: f"{r.get('username','')}<br>Posts: {r['post_count']:,}<br>Btw: {r['betweenness_public']:.4f}",
            axis=1
        ),
        hoverinfo='text',
    ))
    fig.update_layout(
        title='Actores IronMarch: volumen de posts vs betweenness centrality<br>'
              '<sup>Cuadrante superior derecho = máxima influencia (volumen + conectividad)</sup>',
        xaxis_title='Posts totales',
        yaxis_title='Betweenness centrality',
        template='plotly_dark',
        height=550,
    )
    fig.show()

### Hallazgo P1

**Hallazgo:** existe una minoría de usuarios (aproximadamente el 5%) que genera la mayoría
del contenido y ocupa posiciones centrales en la red. Esta minoría se segmenta en dos roles:
- **Líderes ideológicos**: alto betweenness público, alto volumen, vocabulario de nivel filosófico
- **Operadores de red**: alto betweenness en PMs, volumen moderado — coordinación privada

**Evidencia:** betweenness centrality del notebook 02 + perfiles LLM del notebook 03.

**Limitación:** los perfiles LLM son síntesis basadas en texto — no podemos verificar
las identidades reales sin fuentes externas (investigaciones judiciales, doxxing previo).

**Confianza:** 🟡 media — la identificación de rol es hipótesis, los datos de red son verificables.

## 4. Respuesta a P2: Sockpuppets

In [ ]:
# P2: Sockpuppets — cargar resultados de Burrows' Delta del notebook 03

# Intentar cargar la matriz de Delta si fue guardada
delta_path = IM_RESULTS / 'delta_pairs.parquet'

if delta_path.exists():
    pairs_df = pd.read_parquet(delta_path)
    q01 = pairs_df['delta'].quantile(0.01)
    suspicious = pairs_df[pairs_df['delta'] <= q01].sort_values('delta')

    print(f'Pares analizados: {len(pairs_df):,}')
    print(f'Delta medio del corpus: {pairs_df["delta"].mean():.4f}')
    print(f'Umbral de sospecha (percentil 1%): Delta < {q01:.4f}')
    print(f'Pares sospechosos: {len(suspicious)}')
    print()
    print('Top candidatos a sockpuppet (Delta más bajo):')
    print(suspicious[['user_a', 'user_b', 'delta']].head(10).to_string(index=False))

    # Distribución de Delta
    fig = go.Figure()
    fig.add_trace(go.Histogram(
        x=pairs_df['delta'], nbinsx=60,
        marker_color='steelblue', opacity=0.8, name='todos los pares'
    ))
    fig.add_vline(x=q01, line_color='red', line_dash='dash',
                  annotation_text=f'P1% = {q01:.3f}<br>(umbral sospecha)')
    fig.add_vline(x=pairs_df['delta'].quantile(0.10), line_color='orange', line_dash='dash',
                  annotation_text=f'P10% = {pairs_df["delta"].quantile(0.10):.3f}')
    fig.update_layout(
        title="Distribución de Burrows' Delta — IronMarch<br>"
              "<sup>Umbral rojo: percentil 1% — pares más similares estilísticamente</sup>",
        xaxis_title="Delta (↓ = más similar)",
        yaxis_title='Pares',
        template='plotly_dark', height=420,
    )
    fig.show()
else:
    print('Resultados de Delta no encontrados.')
    print('Ejecutar notebook 03_analisis_semantico.ipynb hasta la sección de Burrows Delta.')
    print()
    print('Resultado pre-calculado del análisis original:')
    print('  Par más sospechoso: 255 (userid=0, ~Slavros) + Talleyrand (userid=16)')
    print('  Delta: 0.4637  (el más bajo del corpus)')
    print('  Similitud temporal: 0.9896  (prácticamente idéntico en horario de posteo)')
    print('  Talleyrand: email anicot@elon.edu (Elon University), activo 2011-2014')

### Hallazgo P2

**Hallazgo:** el análisis estilométrico identifica un candidato sólido a cuenta doble:
el usuario `255` (userid=0, probable cuenta de administración de Alexander Slavros)
y el usuario `Talleyrand` muestran el Delta más bajo del corpus y una similitud temporal
de horarios de posteo de 0.99 (casi idéntica).

Talleyrand tiene email `anicot@elon.edu` (Elon University, NC) y fue activo
solo entre 2011 y 2014, tras lo cual desaparece. Esta desaparición abrupta podría
indicar que la cuenta fue abandonada, no necesariamente que era una cuenta doble.

**Evidencia:** Burrows' Delta del notebook 03 + análisis de patrones temporales.

**Limitación:** la estilometría no es prueba conclusiva. En un corpus temáticamente
homogéneo como IronMarch (todos discuten las mismas ideas con el mismo vocabulario),
el poder discriminante del Delta es menor. Requiere corroboración por análisis de contenido.

**Confianza:** 🔴 baja-media — es un lead, no una conclusión.

## 5. Respuesta a P3: Eventos externos y actividad

In [ ]:
# P3: Correlación con eventos externos
# Reproducimos el análisis temporal con todos los datos disponibles

EVENTS = {
    '2011-07-22': 'Breivik (Noruega)',
    '2012-08-05': 'Wisconsin Sikh Temple',
    '2015-01-07': 'Charlie Hebdo',
    '2015-06-17': 'Charleston',
    '2015-11-13': 'Atentados París',
    '2016-07-14': 'Niza',
    '2016-11-08': 'Trump elegido',
    '2017-08-12': 'Charlottesville',
}

if len(posts_clean) > 0 and 'dateline' in posts_clean.columns:
    # Asegurar que dateline es datetime con timezone
    if not pd.api.types.is_datetime64_any_dtype(posts_clean['dateline']):
        posts_clean['dateline'] = pd.to_datetime(posts_clean['dateline'], utc=True, errors='coerce')

    valid = posts_clean.dropna(subset=['dateline'])
    valid = valid[(valid['dateline'].dt.year >= 2011) & (valid['dateline'].dt.year <= 2018)]

    weekly = valid.groupby(valid['dateline'].dt.to_period('W')).size()
    weekly_idx = [p.start_time for p in weekly.index]

    mu = weekly.mean()
    sigma = weekly.std()
    threshold = mu + 2 * sigma

    fig = go.Figure()
    fig.add_trace(go.Scatter(
        x=weekly_idx, y=weekly.values,
        mode='lines', fill='tozeroy',
        line=dict(color='#E94E4E', width=1.5),
        fillcolor='rgba(233,78,78,0.12)',
        name='posts/semana',
    ))
    fig.add_hline(y=threshold, line_dash='dash', line_color='orange',
                  annotation_text=f'µ+2σ = {threshold:.0f}')

    spike_weeks = [(p, weekly[p]) for p in weekly.index if weekly[p] > threshold]
    fig.add_trace(go.Scatter(
        x=[p.start_time for p, _ in spike_weeks],
        y=[v for _, v in spike_weeks],
        mode='markers', marker=dict(color='yellow', size=10, symbol='star'),
        name='spike estadístico',
    ))

    tz = valid['dateline'].dt.tz
    for date_str, label in EVENTS.items():
        dt = pd.Timestamp(date_str, tz='UTC') if tz else pd.Timestamp(date_str)
        fig.add_vline(x=dt, line_color='gold', line_dash='dot', line_width=1, opacity=0.5,
                      annotation_text=label, annotation_textangle=-90,
                      annotation_font_size=9, annotation_font_color='gold')

    fig.update_layout(
        title='IronMarch — actividad semanal y eventos externos 2011–2018',
        xaxis_title='Semana', yaxis_title='Posts/semana',
        template='plotly_dark', height=450,
        legend=dict(orientation='h', y=1.02),
    )
    fig.show()

    print(f'\nActividad semanal — µ={mu:.0f} posts | σ={sigma:.0f} | umbral={threshold:.0f}')
    print(f'Spikes detectados (z>2): {len(spike_weeks)}')
    print('\nSpikes y eventos próximos:')
    for p, count in sorted(spike_weeks, key=lambda x: x[1], reverse=True):
        print(f'  {p.start_time.strftime("%Y-%m-%d")}  {count:,} posts')
else:
    print('Sin datos temporales disponibles.')

### Hallazgo P3

**Hallazgo:** los **atentados de París de noviembre 2015** son el único evento externo
con spike estadístico confirmado — actividad 1.6× sobre la media, índice de activación
de 2.35σ. La respuesta no fue sobre supremacismo racial sino **islamofóbica**:
las menciones de `paris` se multiplicaron ×22, `refugee` ×16, `islam` ×4.

Sorprendentemente, **Trump (nov 2016) y Charlottesville (ago 2017) no generaron spike**.
Para esas fechas el foro estaba en declive activo — coherente con el cierre en noviembre 2017.

**Evidencia:** análisis de z-score semanal del notebook 02 + análisis de frecuencias
de keywords del notebook 02.

**Limitación:** no podemos establecer causalidad. El spike temporal coincide con el atentado,
pero no podemos descartar otras causas concurrentes.

**Confianza:** 🟢 alta — el spike es estadísticamente robusto (z>2) y la evidencia de contenido
(frecuencias de keywords) es consistente.

## 6. Respuesta a P4: Estructura de comunidades

In [ ]:
# P4: Comunidades — visualización del cluster HDBSCAN del embedding

if len(semantic_signals) > 0 and 'hdbscan_cluster' in semantic_signals.columns:
    cluster_dist = semantic_signals['hdbscan_cluster'].value_counts().sort_index()

    # Excluir ruido (-1)
    cluster_real = cluster_dist[cluster_dist.index != -1]

    fig = go.Figure(go.Bar(
        x=[f'Cluster {i}' for i in cluster_real.index],
        y=cluster_real.values,
        marker_color='#E94E4E',
    ))
    fig.update_layout(
        title='Tamaño de clusters HDBSCAN — IronMarch (basado en embeddings)',
        xaxis_title='Cluster', yaxis_title='Usuarios en el cluster',
        template='plotly_dark', height=380,
    )
    fig.show()

    noise_count = cluster_dist.get(-1, 0)
    total = len(semantic_signals)
    print(f'Clusters detectados: {len(cluster_real)}')
    print(f'Usuarios en algún cluster: {cluster_real.sum():,} ({cluster_real.sum()/total*100:.1f}%)')
    print(f'Ruido (sin cluster): {noise_count:,} ({noise_count/total*100:.1f}%)')

    # Mostrar top usuarios por cluster
    if len(posts_clean) > 0:
        post_cnt = posts_clean.groupby('userid').size().reset_index(name='n_posts')
        sigs_with_posts = semantic_signals.merge(post_cnt, on='userid', how='left')

        print('\nTop usuarios por cluster (más prolíficos):')
        for cid in sorted(cluster_real.index):
            members = (
                sigs_with_posts[sigs_with_posts['hdbscan_cluster'] == cid]
                .nlargest(5, 'n_posts')[['username', 'n_posts']]
            )
            names = [f"{r['username']} ({r['n_posts']:,})" for _, r in members.iterrows()]
            print(f'  Cluster {cid}: {" | ".join(names)}')
else:
    print('Señales semánticas no disponibles.')
    print('Ejecutar 03_analisis_semantico.ipynb hasta la sección de UMAP+HDBSCAN.')

### Hallazgo P4

**Hallazgo:** el embedding del notebook 03 identifica entre 8 y 12 subcomunidades
(clusters HDBSCAN) con vocabulario y patrones de escritura distintos. Estos clusters
no coinciden exactamente con los subforos del foro — emergen del contenido real.

Clusters típicos detectados: neonazis filosóficos (Evola, Spengler), aceleracionistas
(Atomwaffen), discutidores de eventos de actualidad, y miembros con pocas publicaciones
sin perfil ideológico claro.

**Evidencia:** HDBSCAN sobre embeddings qwen3-embedding del notebook 03.

**Limitación:** el número de clusters y su composición dependen de los parámetros de
HDBSCAN. Pequeños cambios en `min_cluster_size` pueden alterar los resultados.

**Confianza:** 🟡 media — la existencia de clusters es robusta; la interpretación
de qué representa cada cluster requiere lectura manual de posts representativos.

## 7. Convergencia de señales por actor

Aquí hacemos lo más valioso del análisis multi-técnica: ver si **diferentes técnicas
independientes señalan a los mismos actores** como relevantes.

Si un usuario aparece como:
- Top betweenness en la red (notebook 02)
- Top broker en PMs (notebook 02)
- Entidades NER muy específicas (notebook 03)
- Cluster ideológico diferenciado (notebook 03)

...entonces la evidencia de su importancia es mucho más sólida que si solo aparece en una técnica.

In [ ]:
# Tabla de convergencia: una fila por usuario, una columna por señal
# Normalizar cada señal a escala 0-1 para poder comparar

def normalize(series):
    """Normalizar una serie a escala 0-1. Maneja NaN sin problema."""
    mn, mx = series.min(), series.max()
    if mx == mn: return series * 0  # Todos iguales → todos cero
    return (series - mn) / (mx - mn)

if len(actors_table) > 0:
    convergence = actors_table.copy()

    # Normalizar señales disponibles
    signals_to_norm = []
    if 'post_count' in convergence.columns:
        convergence['score_volume'] = normalize(convergence['post_count'].fillna(0))
        signals_to_norm.append('score_volume')
    if 'betweenness_public' in convergence.columns:
        convergence['score_btw_pub'] = normalize(convergence['betweenness_public'].fillna(0))
        signals_to_norm.append('score_btw_pub')
    if 'betweenness_pm' in convergence.columns:
        convergence['score_btw_pm'] = normalize(convergence['betweenness_pm'].fillna(0))
        signals_to_norm.append('score_btw_pm')

    if signals_to_norm:
        # Score combinado: promedio de las señales normalizadas disponibles
        convergence['combined_score'] = convergence[signals_to_norm].mean(axis=1)
        top_convergence = convergence.nlargest(20, 'combined_score')

        # Gráfica de radar / heatmap de señales
        disp_cols = ['username'] + signals_to_norm + ['combined_score']
        disp_cols = [c for c in disp_cols if c in top_convergence.columns]

        # Heatmap de señales normalizadas
        heat_df = top_convergence[signals_to_norm].fillna(0)
        heat_df.index = top_convergence['username'].fillna(top_convergence['userid']).tolist()

        col_labels = {
            'score_volume':  'Volumen posts',
            'score_btw_pub': 'Betweenness público',
            'score_btw_pm':  'Betweenness PMs',
        }

        fig = go.Figure(go.Heatmap(
            z=heat_df.values,
            x=[col_labels.get(c, c) for c in heat_df.columns],
            y=heat_df.index.tolist(),
            colorscale='YlOrRd',
            hovertemplate='%{y}<br>%{x}: %{z:.3f}<extra></extra>',
            colorbar=dict(title='Score<br>normalizado'),
        ))
        fig.update_layout(
            title='Convergencia de señales por actor — top 20 (0=mínimo, 1=máximo)',
            template='plotly_dark',
            height=max(400, 25 * len(heat_df)),
            margin=dict(l=160, r=20, t=60, b=40),
        )
        fig.show()

        print('\nScore combinado de los 10 actores con mayor convergencia de señales:')
        print(top_convergence[disp_cols].head(10).to_string(index=False))
else:
    print('No hay suficientes datos para calcular convergencia.')

## 8. Tabla resumen de actores clave

Esta es la tabla de síntesis final. Integra todas las señales disponibles para cada
actor clave en un formato que puede entenderse sin conocimientos técnicos.

In [ ]:
# Tabla resumen ejecutiva
if len(actors_table) > 0:
    summary_table = actors_table.nlargest(15, 'post_count').copy()

    # Columnas disponibles para mostrar
    show_cols = ['username', 'post_count']
    col_labels = {'username': 'Usuario', 'post_count': 'Posts'}

    optional_cols = [
        ('betweenness_public', 'Betweenness público'),
        ('betweenness_pm',     'Betweenness PMs'),
        ('hdbscan_cluster',    'Cluster semántico'),
        ('role',               'Rol (LLM)'),
        ('radicalization_level', 'Radicalización'),
        ('min_delta',          'Min Delta estilo'),
    ]
    for col, label in optional_cols:
        if col in summary_table.columns:
            show_cols.append(col)
            col_labels[col] = label

    display_df = summary_table[show_cols].rename(columns=col_labels)

    # Mostrar como tabla Plotly para mejor presentación
    fig = go.Figure(go.Table(
        header=dict(
            values=list(display_df.columns),
            fill_color='#2C2C2C',
            font=dict(color='white', size=11),
            align='left',
        ),
        cells=dict(
            values=[display_df[col].fillna('—').astype(str) for col in display_df.columns],
            fill_color='#1C1C1C',
            font=dict(color='white', size=10),
            align='left',
        )
    ))
    fig.update_layout(
        title='Resumen de actores clave — IronMarch',
        template='plotly_dark', height=500,
    )
    fig.show()
else:
    print('Sin datos suficientes para tabla resumen.')

## 9. Conclusiones

### ¿Qué respondimos?

✅ **P1 — Actores clave**: identificamos una élite de ~5% de usuarios que genera la mayoría
del contenido con roles diferenciados (líderes filosóficos, operadores de red). El análisis
converge entre técnicas de red y semánticas.

🟡 **P2 — Sockpuppets**: encontramos un candidato sólido (par con Delta más bajo + horario
idéntico), pero la estilometría no es prueba conclusiva en un corpus homogéneo.

✅ **P3 — Eventos externos**: los atentados de París (nov 2015) generaron el único spike
estadístico confirmado. La respuesta fue islamofóbica, no relacionada con el supremacismo
racial que domina el ideario cotidiano del foro.

🟡 **P4 — Comunidades**: existen subcomunidades semánticas diferenciadas que no coinciden
exactamente con la estructura de subforos oficial. Los clusters de embeddings revelan
afinidades de estilo e ideas que trascienden las categorías administrativas.

### ¿Qué queda abierto?

- **Verificación de identidades**: las identidades propuestas (sockpuppets, líderes) requieren
  corroboración con fuentes externas (investigaciones judiciales previas, OSINT).

- **Análisis de contenido de PMs**: tenemos 55K mensajes privados casi sin explorar.
  Un análisis específico de PMs podría revelar coordinación privada y operaciones que
  no eran visibles en el foro público.

- **Rastreo de actores fuera del foro**: algunos usuarios con emails conocidos
  (especialmente .edu) podrían rastrearse en otras plataformas.

- **Análisis en alemán y ruso**: ~15% de los posts no son en inglés y no fueron
  analizados con el pipeline semántico actual.

### ¿Por qué este dataset es excepcional?

La mayoría de leaks de foros solo contienen posts públicos. IronMarch incluye **mensajes privados**,
que son la capa de coordinación que normalmente queda invisible. La existencia de **ground truth**
(miembros identificados en investigaciones posteriores, el fundador doxxeado) permite validar
el pipeline computacional de una manera que rara vez es posible.

## 10. Exportación del informe

### Opción A: Exportar como HTML interactivo

El comando de abajo convierte el notebook en un archivo HTML que incluye
todas las gráficas interactivas de Plotly. Puede abrirse en cualquier navegador
sin necesidad de Python o Jupyter.

```bash
jupyter nbconvert --to html 04_sintesis_informe.ipynb --output informe_ironmarch.html
```

### Opción B: Exportar como PDF

Requiere tener `pandoc` y una instalación de LaTeX instalada.
Las gráficas interactivas se convierten en imágenes estáticas.

```bash
jupyter nbconvert --to pdf 04_sintesis_informe.ipynb --output informe_ironmarch.pdf
```

### Opción C: Exportar como HTML ejecutable completo

Este formato incluye las salidas ya calculadas sin necesidad de re-ejecutar el notebook:

```bash
jupyter nbconvert --to html --execute 04_sintesis_informe.ipynb
```

In [ ]:
import subprocess

# Exportar este notebook como HTML
# Ejecutar esta celda genera el informe en HTML directamente desde Python
output_name = IM_RESULTS / 'informe_ironmarch.html'

result = subprocess.run(
    ['jupyter', 'nbconvert', '--to', 'html',
     '04_sintesis_informe.ipynb',
     '--output', str(output_name)],
    capture_output=True, text=True
)

if result.returncode == 0:
    print(f'Informe HTML generado: {output_name}')
    print(f'Tamaño: {output_name.stat().st_size / 1024:.0f} KB')
else:
    print('Error al generar HTML:')
    print(result.stderr)

In [ ]:
# Guardar tabla resumen de actores como CSV (para compartir con no-técnicos)
if len(actors_table) > 0:
    csv_path = IM_RESULTS / 'actores_clave_resumen.csv'
    actors_table.nlargest(50, 'post_count').to_csv(csv_path, index=False)
    print(f'CSV guardado: {csv_path}')

print('\n=== ANÁLISIS COMPLETO — IronMarch ===')
print('Notebooks ejecutados:')
print('  00_reconocimiento.ipynb     — Exploración y preguntas de investigación')
print('  01_ingenieria_datos.ipynb   — Limpieza y preparación')
print('  02_analisis_estructural.ipynb — Red, temporal, TF-IDF')
print('  03_analisis_semantico.ipynb — Embeddings, BERTopic, NER, Delta')
print('  04_sintesis_informe.ipynb   — Integración y conclusiones')
print()
print('Archivos generados en results/ironmarch/:')
for f in sorted(IM_RESULTS.iterdir()):
    if not f.name.startswith('.'):
        size_kb = f.stat().st_size / 1024
        print(f'  {f.name:<40} {size_kb:>8.0f} KB')